# Phase 8.5: Curriculum Learning — ABC School for FLUX (Colab)
## Teaching FLUX to spell: bytes → bigrams → words → phrases → sentences → real text

**Goal:** Train the WaveDecoder through a 6-stage curriculum so FLUX can generate coherent text, not gibberish.

**Platform:** Google Colab Pro (A100 recommended). Uses Google Drive for persistent checkpoint storage.

### What this notebook does

1. Mount Google Drive + clone/pull FLUX repo
2. Install dependencies + setup
3. Initialize logger, detect hardware, load secrets
4. Build FLUXLarge from Phase 8 checkpoint
5. Run 6-stage curriculum training
6. Save checkpoint (phase8_5.phase.pt) + upload to HuggingFace
7. Test 1: Word spelling accuracy
8. Test 2: Sentence coherence
9. Test 3: Generation quality vs Phase 8
10. Demo 1: Stage-by-stage generation showcase
11. Demo 2: FLUX vs GPT-2 generation
12. View results + full log
13. Final upload (logs → HF + GitHub)
14. Save artifacts to Google Drive

### Curriculum Stages

| Stage | Material | Max Steps | Goal |
|-------|----------|-----------|------|
| 1 | Printable ASCII bytes | 200 | Learn byte distribution |
| 2 | Common bigrams/trigrams | 500 | Learn byte co-occurrence |
| 3 | Top-1000 English words | 2000 | Learn to spell words |
| 4 | Common phrases | 3000 | Learn multi-word patterns |
| 5 | Simple sentences | 3000 | Learn grammar + punctuation |
| 6 | Real OpenWebText | 5000 | Graduate to real text |

### Secrets Required

- `HF_TOKEN`: Set via Colab Secrets (key icon in left sidebar)

---
## Cell 1: Mount Google Drive + Clone / Pull Repository

In [ ]:
import os
import sys
import subprocess
import importlib
from pathlib import Path

REPO_URL = "https://github.com/Unseengap/FLUX.git"

# ─────────────────────────────────────────────
# 0. Mount Google Drive for persistent storage
# ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_FLUX = Path('/content/drive/MyDrive/FLUX')
DRIVE_FLUX.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINTS = DRIVE_FLUX / 'checkpoints'
DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
DRIVE_LOGS = DRIVE_FLUX / 'logs'
DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
print(f"  ✓ Google Drive mounted — persistent storage at {DRIVE_FLUX}")

WORK_DIR = Path("/content/FLUX")

# ─────────────────────────────────────────────
# 1. Clone or Pull — always get latest code
# ─────────────────────────────────────────────
if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
    print("ℹ Repo already exists — pulling latest changes...")
    subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL],
                   cwd=str(WORK_DIR), capture_output=True, text=True)
    subprocess.run(['git', 'checkout', '.'], cwd=str(WORK_DIR),
                   capture_output=True, text=True)
    subprocess.run(['git', 'clean', '-fd'], cwd=str(WORK_DIR),
                   capture_output=True, text=True)
    result = subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'],
                           cwd=str(WORK_DIR), capture_output=True, text=True)
    print(result.stdout.strip() or result.stderr.strip())
    head = subprocess.run(['git', 'log', '--oneline', '-3'],
                         cwd=str(WORK_DIR), capture_output=True, text=True)
    print(f"\n  Latest commits:\n{head.stdout.strip()}")
    print("\n✓ Pulled & reset to latest origin/main")
else:
    if WORK_DIR.exists():
        import shutil
        shutil.rmtree(str(WORK_DIR))
    print(f"ℹ Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)
    print("✓ Cloned successfully")

os.chdir(str(WORK_DIR))
print(f"\nWorking directory: {os.getcwd()}")

# ─────────────────────────────────────────────
# 1b. Symlink Drive → local
# ─────────────────────────────────────────────
local_ckpt = WORK_DIR / 'checkpoints'
if local_ckpt.is_symlink():
    local_ckpt.unlink()
elif local_ckpt.exists():
    import shutil
    for f in local_ckpt.glob('*.pt'):
        dst = DRIVE_CHECKPOINTS / f.name
        if not dst.exists():
            shutil.copy2(str(f), str(dst))
    shutil.rmtree(str(local_ckpt))
local_ckpt.symlink_to(DRIVE_CHECKPOINTS)
print(f"  ✓ checkpoints/ → {DRIVE_CHECKPOINTS}")

local_logs = WORK_DIR / 'logs'
if local_logs.is_symlink():
    local_logs.unlink()
elif local_logs.exists():
    import shutil
    for f in local_logs.glob('*.log'):
        dst = DRIVE_LOGS / f.name
        if not dst.exists():
            shutil.copy2(str(f), str(dst))
    shutil.rmtree(str(local_logs))
local_logs.symlink_to(DRIVE_LOGS)
print(f"  ✓ logs/ → {DRIVE_LOGS}")

# ─────────────────────────────────────────────
# 2. Flush cached modules
# ─────────────────────────────────────────────
_stale = [m for m in sys.modules if any(
    m.startswith(p) for p in (
        'flux_large', 'flux_model', 'flux_generate', 'flux_trainer',
        'train_openwebtext', 'benchmark_gpt2', 'wave_decoder',
        'curriculum_data', 'curriculum_trainer', 'train_curriculum',
        'working_memory', 'episodic_memory', 'semantic_memory',
        'memory_router', 'consolidation', 'train_memory',
        'cgn', 'manifold', 'causal_graph', 'multi_timescale',
        'thermodynamic', 'temperature', 'energy_functions', 'online_learner',
        'cse', 'field', 'flux_utils', 'wave_types', 'interference',
        'attractor', 'field_ops', 'train_', 'demo_', 'test_',
        'gravity', 'mass_tracker', 'spatial_index', 'negative_mass',
    )
)]
for m in _stale:
    del sys.modules[m]
if _stale:
    print(f"  ✓ Flushed {len(_stale)} cached modules")

subprocess.run(['python', 'setup.py'], cwd=str(WORK_DIR),
               capture_output=True, text=True)

# Verify Phase 8.5 modules
_ct_path = WORK_DIR / 'phases' / 'phase8_5' / 'curriculum_trainer.py'
assert _ct_path.exists(), f"ERROR: {_ct_path} not found!"
print(f"  ✓ curriculum_trainer.py verified")

_cd_path = WORK_DIR / 'phases' / 'phase8_5' / 'curriculum_data.py'
assert _cd_path.exists(), f"ERROR: {_cd_path} not found!"
print(f"  ✓ curriculum_data.py verified")

print("✓ setup.py refreshed")
print(f"\n  Google Drive checkpoints:")
for f in sorted(DRIVE_CHECKPOINTS.glob('*.pt')):
    print(f"    {f.name} ({f.stat().st_size/1e6:.1f} MB)")
if not list(DRIVE_CHECKPOINTS.glob('*.pt')):
    print(f"    (no checkpoints yet)")

---
## Cell 2: Install Dependencies

In [ ]:
!pip install -q datasets rich faiss-cpu huggingface_hub matplotlib transformers scipy
!python setup.py

---
## Cell 3: Initialize Logger, Detect Hardware & Load Secrets

In [ ]:
import sys, torch, platform, importlib
from pathlib import Path

# ── Add project paths ──
sys.path.insert(0, str(Path.cwd()))
for _phase in ['phase1', 'phase2', 'phase3', 'phase4', 'phase5',
               'phase6', 'phase7', 'phase8', 'phase8_5']:
    sys.path.insert(0, str(Path.cwd() / 'phases' / _phase))

# ── Force-reload all project modules ──
for _mod_name in list(sys.modules.keys()):
    if any(_mod_name.startswith(p) for p in (
        'flux_utils', 'flux_model', 'flux_large', 'wave_decoder',
        'curriculum_data', 'curriculum_trainer', 'train_curriculum',
        'train_openwebtext', 'benchmark_gpt2',
        'working_memory', 'episodic_memory', 'semantic_memory',
        'memory_router', 'consolidation',
        'cgn', 'manifold', 'causal_graph', 'multi_timescale',
        'thermodynamic', 'temperature', 'energy_functions', 'online_learner',
        'cse', 'field', 'wave_types', 'interference', 'attractor', 'field_ops',
        'gravity', 'mass_tracker', 'spatial_index', 'negative_mass',
    )):
        del sys.modules[_mod_name]

from flux_utils import (
    get_device, get_hardware_info, PhaseLogger, _resolve_hf_token,
    load_checkpoint, save_checkpoint, verify_checkpoint_chain,
    upload_checkpoint_to_hf, upload_logs_to_hf, git_commit_and_push,
    PhaseResults, checkpoint_exists,
)

# ── Initialize Logger ──
log = PhaseLogger(phase=8)
log.separator("Phase 8.5: Curriculum Learning — ABC School (Colab)")
log.cell_start("Cell 3 — Hardware & Secrets")

# ── Detect hardware ──
DEVICE = get_device()
hw = get_hardware_info()
log.info(f"Device: {DEVICE}")
log.info(f"PyTorch: {torch.__version__}")
for k, v in hw.items():
    print(f"  {k}: {v}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n  🚀 GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

# ── Load HuggingFace token ──
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        print(f"  ✓ HF_TOKEN loaded from Colab Secrets")
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = _resolve_hf_token()
if HF_TOKEN:
    log.success("HuggingFace token loaded")
else:
    print("  ⚠ No HF token — add HF_TOKEN to Colab Secrets")

# ── Drive status ──
drive_ckpts = list(Path('/content/drive/MyDrive/FLUX/checkpoints').glob('*.pt'))
if drive_ckpts:
    print(f"\n  📁 Drive checkpoints ({len(drive_ckpts)}):")
    for f in sorted(drive_ckpts):
        print(f"    {f.name} ({f.stat().st_size/1e6:.1f} MB)")

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

---
## Cell 4: Build FLUXLarge from Phase 8 Checkpoint

Loads the Phase 8 trained model (including WaveDecoder) as the starting point for curriculum training. The decoder has basic training from OpenWebText — the curriculum will refine it.

In [ ]:
log.cell_start("Cell 4 — Build FLUXLarge")

import torch
import importlib

# Force-reimport
for _m in ['cse', 'field', 'gravity', 'mass_tracker', 'spatial_index', 'negative_mass',
           'thermodynamic', 'temperature', 'energy_functions', 'online_learner',
           'cgn', 'manifold', 'causal_graph', 'multi_timescale',
           'working_memory', 'episodic_memory', 'semantic_memory',
           'memory_router', 'consolidation',
           'flux_model', 'flux_large', 'wave_decoder',
           'curriculum_data', 'curriculum_trainer', 'train_curriculum',
           'train_openwebtext', 'benchmark_gpt2']:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

from flux_large import FLUXLarge, FLUX_LARGE_CONFIG

# ── Load model ──
print("  Loading FLUXLarge...")
if checkpoint_exists(8):
    print("  ✓ Phase 8 checkpoint found — loading trained model")
    model = FLUXLarge.from_phase8_checkpoint(device=DEVICE)
else:
    print("  ℹ No Phase 8 checkpoint — loading from Phase 7")
    model = FLUXLarge.from_phase7_checkpoint(device=DEVICE)

# ── Smoke test ──
print("\n  Running smoke test...")
response = model.forward("Curriculum learning smoke test", learn=False)
assert response.wave is not None
print(f"  ✓ Smoke test passed ({response.latency_ms:.1f}ms)")

# ── Stats ──
stats = model.get_stats()
print(f"\n  FLUXLarge: {stats.total_params:,} params")
print(f"  Decoder:   {sum(p.numel() for p in model.decoder.parameters()):,} params")

# ── Quick pre-curriculum generation ──
print(f"\n  Pre-curriculum generation:")
for prompt in ["The ", "I think ", "She said "]:
    gen = model.generate(prompt, max_length=30, temperature=0.7)
    print(f"    '{prompt}' → '{gen[len(prompt):]}'")

log.cell_end("Cell 4 — Build FLUXLarge", "PASS")

---
## Cell 5: Curriculum Training — 6 Stages

The ABC School:
1. **Bytes** — learn to produce printable ASCII (like learning the alphabet)
2. **Bigrams** — learn common 2-3 letter combinations (like phonics)
3. **Words** — learn to spell the 1000 most common English words
4. **Phrases** — learn common multi-word patterns ("in the", "it is")
5. **Sentences** — learn grammar, punctuation, structure
6. **Real text** — graduate to OpenWebText

Each stage monitors decoder loss and runs advancement tests.

In [ ]:
log.cell_start("Cell 5 — Curriculum Training")

import time
from pathlib import Path

_PHASE85_CKPT = Path('checkpoints') / 'phase8_5.phase.pt'
_PHASE85_FRESH = not _PHASE85_CKPT.exists()

if not _PHASE85_FRESH:
    print("⏩ Phase 8.5 checkpoint found — loading instead of training")
    import torch
    ckpt85 = torch.load(str(_PHASE85_CKPT), map_location='cpu')
    curriculum_metrics = ckpt85.get('metrics', {})
    curriculum_state = ckpt85.get('curriculum_state', {})
    
    # Load decoder state into model
    if 'decoder_state_dict' in ckpt85:
        model.decoder.load_state_dict(ckpt85['decoder_state_dict'])
        print("  ✓ Decoder state loaded from Phase 8.5 checkpoint")
    for key in ['wave_to_field_state', 'field_state_dict', 'output_head_state']:
        if key in ckpt85:
            try:
                attr = key.replace('_state_dict', '').replace('_state', '')
                getattr(model, attr).load_state_dict(ckpt85[key])
            except Exception:
                pass
    model = model.to(DEVICE)
    
    print(f"  Stages completed: {curriculum_state.get('stages_completed', '?')}")
    print(f"  Total steps:      {curriculum_state.get('total_steps', '?')}")
    history = curriculum_state.get('stage_history', [])
    for s in history:
        print(f"    Stage {s['stage']}: {s['name']:<24} loss={s['avg_loss']:.4f} acc={s['accuracy']:.1%}")

else:
    print("── Starting Curriculum Training ──\n")
    start_time = time.time()

    from curriculum_trainer import CurriculumTrainer
    from train_openwebtext import load_openwebtext_subset

    # ── Load OpenWebText for Stage 6 ──
    print("Loading OpenWebText for Stage 6...")
    owt_texts = load_openwebtext_subset(max_docs=1000)
    print(f"  ✓ {len(owt_texts)} documents loaded\n")

    # ── Run curriculum ──
    trainer = CurriculumTrainer(
        model=model,
        log=log,
        openwebtext_texts=owt_texts,
        verbose=True,
    )

    curriculum_result = trainer.run_curriculum(start_stage=1)
    elapsed = time.time() - start_time

    # ── Build metrics ──
    curriculum_metrics = {
        'stages_completed': curriculum_result.stages_completed,
        'total_steps': curriculum_result.total_steps,
        'total_time_seconds': elapsed,
        'final_stage': curriculum_result.final_stage,
    }
    for r in curriculum_result.stage_results:
        curriculum_metrics[f'stage{r.stage}_loss'] = r.avg_loss
        curriculum_metrics[f'stage{r.stage}_accuracy'] = r.accuracy
        curriculum_metrics[f'stage{r.stage}_steps'] = r.steps_taken

    curriculum_state = {
        'stages_completed': curriculum_result.stages_completed,
        'final_stage': curriculum_result.final_stage,
        'total_steps': curriculum_result.total_steps,
        'stage_history': [
            {
                'stage': r.stage, 'name': r.name,
                'steps': r.steps_taken, 'avg_loss': r.avg_loss,
                'min_loss': r.min_loss, 'accuracy': r.accuracy,
                'advanced': r.advanced, 'time': r.time_seconds,
            }
            for r in curriculum_result.stage_results
        ],
    }

    log.success(f"Curriculum complete: {curriculum_result.total_steps} steps in {elapsed:.1f}s")
    print(f"\n  ── Post-Curriculum Generation ──")
    for prompt in ["The ", "I think ", "She said ", "In the morning "]:
        gen = model.generate(prompt, max_length=30, temperature=0.7)
        print(f"    '{prompt}' → '{gen[len(prompt):]}'")

log.cell_end("Cell 5 — Curriculum Training", "PASS")

---
## Cell 6: Save Checkpoint & Upload to HuggingFace

In [ ]:
log.cell_start("Cell 6 — Save & Upload Checkpoint")

from datetime import datetime
import torch

_PHASE85_CKPT = Path('checkpoints') / 'phase8_5.phase.pt'

if _PHASE85_CKPT.exists() and not _PHASE85_FRESH:
    size_mb = _PHASE85_CKPT.stat().st_size / 1e6
    print(f"  ⏩ Checkpoint already saved — skipping")
    print(f"     {_PHASE85_CKPT} ({size_mb:.1f} MB)")
else:
    # Build checkpoint state
    checkpoint_state = {
        'phase': 8.5,
        'timestamp': datetime.now().isoformat(),
        'config': model.config,
        'learning_steps': model._learning_steps,
        'curriculum_state': curriculum_state,
        # Component states
        'cse_state_dict': model.cse.state_dict(),
        'field_state_dict': model.field.state_dict(),
        'gr_state': model.gr.save_state(),
        'tl_state': model.tl.save_state(),
        'cgn_state': model.cgn.save_state(),
        'causal_graph_state': model.causal_graph.save_state(),
        'working_memory_state': model.working_memory.state_dict_memory(),
        'episodic_memory_state': model.episodic_memory.save_state(),
        'semantic_memory_state': model.semantic_memory.save_state(),
        'router_state': model.memory_router.save_state(),
        'wave_to_field_state': model.wave_to_field.state_dict(),
        'field_to_wave_state': model.field_to_wave.state_dict(),
        'output_head_state': model.output_head.state_dict(),
        'decoder_state_dict': model.decoder.state_dict(),
        'metrics': curriculum_metrics,
    }

    torch.save(checkpoint_state, str(_PHASE85_CKPT))

    if _PHASE85_CKPT.exists():
        size_mb = _PHASE85_CKPT.stat().st_size / 1e6
        log.success(f"Checkpoint saved: {_PHASE85_CKPT} ({size_mb:.1f} MB)")
        print(f"  ✓ Checkpoint saved: {_PHASE85_CKPT} ({size_mb:.1f} MB)")
    else:
        log.error("Checkpoint NOT found — save may have failed")

    # Upload to HuggingFace (phase 8 slot — 8.5 shares the repo)
    try:
        from huggingface_hub import HfApi
        if HF_TOKEN:
            api = HfApi(token=HF_TOKEN)
            api.upload_file(
                path_or_fileobj=str(_PHASE85_CKPT),
                path_in_repo='checkpoints/phase8_5.phase.pt',
                repo_id='UnseenGAP/FLUX',
                repo_type='model',
            )
            log.success("Checkpoint uploaded to HuggingFace Hub")
            print("  ✓ Uploaded to HuggingFace Hub")
        else:
            print("  ⚠ No HF token — upload skipped")
    except Exception as e:
        print(f"  ⚠ Upload failed: {e}")

    upload_logs_to_hf(phase=8, hf_token=HF_TOKEN)

log.cell_end("Cell 6 — Save & Upload Checkpoint", "PASS")

---
## Cells 7–9: Tests

| Test | Description | Pass Criteria |
|------|-------------|---------------|
| Test 1 | Word Spelling Accuracy | > 50% byte accuracy, >= 30 perfect words |
| Test 2 | Sentence Coherence | > 30% coherent, avg length > 5 |
| Test 3 | Generation vs Phase 8 | Coherence >= baseline |

In [ ]:
log.cell_start("Cell 7 — Test 1: Word Spelling")

import os
_phase85_dir = str(Path.cwd() / 'phases' / 'phase8_5')
_orig_dir = os.getcwd()
os.chdir(_phase85_dir)

%run test_phase8_5_test1.py

os.chdir(_orig_dir)
log.cell_end("Cell 7 — Test 1: Word Spelling", "PASS")

In [ ]:
log.cell_start("Cell 8 — Test 2: Sentence Coherence")

os.chdir(_phase85_dir)

%run test_phase8_5_test2.py

os.chdir(_orig_dir)
log.cell_end("Cell 8 — Test 2: Sentence Coherence", "PASS")

In [ ]:
log.cell_start("Cell 9 — Test 3: Generation vs Phase 8")

os.chdir(_phase85_dir)

%run test_phase8_5_test3.py

os.chdir(_orig_dir)
log.cell_end("Cell 9 — Test 3: Generation vs Phase 8", "PASS")

---
## Cells 10–11: Demos

| Demo | Description |
|------|-------------|
| Demo 1 | Stage-by-stage generation showcase |
| Demo 2 | Curriculum FLUX vs GPT-2 |

In [ ]:
log.cell_start("Cell 10 — Demo 1: Generation Showcase")

os.chdir(_phase85_dir)

%run demo_phase8_5_demo1.py

os.chdir(_orig_dir)
log.cell_end("Cell 10 — Demo 1: Generation Showcase", "PASS")

In [ ]:
log.cell_start("Cell 11 — Demo 2: FLUX vs GPT-2")

os.chdir(_phase85_dir)

%run demo_phase8_5_demo2.py

os.chdir(_orig_dir)
log.cell_end("Cell 11 — Demo 2: FLUX vs GPT-2", "PASS")

---
## Cell 12: View Results & Log

In [ ]:
log.cell_start("Cell 12 — Results & Log")

# ── Generate RESULTS_PHASE_8_5.md ──
results_path = Path('phases/phase8_5/RESULTS_PHASE_8_5.md')
if results_path.exists():
    from IPython.display import Markdown, display
    display(Markdown(results_path.read_text()))
else:
    results = PhaseResults(phase=8, component_name="Phase 8.5 — Curriculum Learning")
    
    history = curriculum_state.get('stage_history', [])
    for s in history:
        results.add_test(
            f"Stage {s['stage']}: {s['name']}",
            passed=s.get('advanced', False),
            score=f"loss={s['avg_loss']:.4f} acc={s['accuracy']:.1%}",
            threshold=f"steps={s['steps']}",
        )
    results.add_metric("total_steps", curriculum_metrics.get('total_steps', 0))
    results.add_metric("stages_completed", curriculum_metrics.get('stages_completed', 0))
    results.add_metric("total_time", f"{curriculum_metrics.get('total_time_seconds', 0):.1f}s")
    
    results.save(custom_path=str(results_path))
    from IPython.display import Markdown, display
    display(Markdown(results_path.read_text()))

# ── Full log ──
print("\n" + "=" * 60)
print("  FULL PHASE 8.5 LOG")
print("=" * 60)
print(log.get_log_contents())

log.cell_end("Cell 12 — Results & Log", "PASS")

---
## Cell 13: Final Upload

In [ ]:
log.cell_start("Cell 13 — Final Upload")

upload_logs_to_hf(phase=8, hf_token=HF_TOKEN)
log.success("Logs uploaded to HuggingFace Hub")

git_commit_and_push(
    files=[
        'logs/phase8.log',
        'phases/phase8_5/',
        'results/',
    ],
    message='Phase 8.5: Curriculum Learning — ABC School [auto-commit from Colab]',
    repo_dir=str(WORK_DIR),
)

log.cell_end("Cell 13 — Final Upload", "PASS")

print("\n" + "=" * 60)
print("✓ PHASE 8.5 COMPLETE")
print("=" * 60)
print(f"  Checkpoint: checkpoints/phase8_5.phase.pt")
print(f"  HuggingFace: https://huggingface.co/UnseenGAP/FLUX")
print(f"  Source: https://github.com/Unseengap/FLUX")
print(f"  Drive: /content/drive/MyDrive/FLUX/checkpoints/")
print("=" * 60)

---
## Cell 14: Save Artifacts to Google Drive

In [ ]:
log.cell_start("Cell 14 — Save Drive Artifacts")

import shutil

DRIVE_FLUX = Path('/content/drive/MyDrive/FLUX')
output_dir = DRIVE_FLUX / 'output' / 'phase8_5'
output_dir.mkdir(parents=True, exist_ok=True)

# Checkpoint
src = WORK_DIR / 'checkpoints' / 'phase8_5.phase.pt'
if src.exists():
    shutil.copy2(str(src), str(output_dir / src.name))
    print(f"  ✓ Checkpoint: {src.name} ({src.stat().st_size/1e6:.1f} MB)")

# Results and logs
for fpath in [
    WORK_DIR / 'phases' / 'phase8_5' / 'RESULTS_PHASE_8_5.md',
    WORK_DIR / 'logs' / 'phase8.log',
]:
    if fpath.exists():
        shutil.copy2(str(fpath), str(output_dir / fpath.name))
        print(f"  ✓ {fpath.name}")

print(f"\n  Saved artifacts in {output_dir}:")
for f in sorted(output_dir.iterdir()):
    size = f.stat().st_size
    unit = 'MB' if size > 1e6 else 'KB'
    val = size / 1e6 if size > 1e6 else size / 1e3
    print(f"    {f.name:40s} {val:8.1f} {unit}")

log.cell_end("Cell 14 — Save Drive Artifacts", "PASS")

print("\n" + "=" * 60)
print("ALL DONE — Phase 8.5: Curriculum Learning (Colab)")
print("=" * 60)
print(f"  Checkpoints SAFE on Google Drive!")
print(f"  Path: /content/drive/MyDrive/FLUX/")
print("=" * 60)

---
## Acceptance Criteria

| Criterion | Target | Method |
|-----------|--------|--------|
| All 6 stages complete | Training finishes | Cell 5 |
| Word spelling accuracy | > 50% byte accuracy | Test 1 |
| Sentence coherence | > 30% readable | Test 2 |
| Generation vs Phase 8 | Coherence >= baseline | Test 3 |
| Demo shows real words | Not gibberish | Demo 1 |
| Checkpoint saved | phase8_5.phase.pt | Cell 6 |
| RESULTS_PHASE_8_5.md generated | File exists | Cell 12 |

### Curriculum Concept

```
Stage 1: a b c d e f g h ...     (individual bytes)
Stage 2: th he in er an ...       (common pairs)
Stage 3: the and for with ...     (common words)
Stage 4: in the, of the, it is   (common phrases)
Stage 5: The man sees the dog.    (simple sentences)
Stage 6: Real OpenWebText text    (graduation!)
```

### Architecture (unchanged from Phase 8)

```
text → CSE(encode) → [seq, 432] wave
     → wave_to_field → [768] field input
     → GR + CGN + Field → [768] merged context
     → WaveDecoder(GRU) → bytes one-at-a-time
```

The WaveDecoder is the only component being trained.
The curriculum teaches it HOW to spell what the field already knows.